# 05 – Electorate-level analysis
One row per electorate, party vote only. `VOTE_SCOPE` / `COUNT_STAGE` choose the basis: currently **all votes, final 2026 count**
(`data/raw/ge2026_final/out.csv`, special votes included, like 2023's all votes). Use `"ordinary"` / `"preliminary"` for the
ordinary-votes-only view. Needs notebook 01 and 02 only: no geocoding.

2023 → 2026 change is shown for every electorate, matched by name. The 10 seats renamed in the 2025 boundary review
are matched to their main 2023 predecessor (Commission's list, `electorates.PREDECESSOR`). Many seats were also
redrawn: the `overlap` column says how much of the 2026 seat came from the 2023 seat it is compared with
(100 = same area), so treat swings in low-overlap seats with care.
Change `FOCUS` to look at any party.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import vote_scope

FOCUS = "Green Party"
VOTE_SCOPE = "all"    # "ordinary" (advance + election day) or "all" (adds special + overseas votes)
COUNT_STAGE = "final"  # 2026 count: "preliminary" or "final" (data/raw/ge2026_final)
TOP_N = 15             # rows in the ranking charts
HEATMAP_PARTIES = 8    # biggest parties nationally in 2026
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 100)

pv = vote_scope.load("stg_party_votes", VOTE_SCOPE, COUNT_STAGE)
print(vote_scope.describe(VOTE_SCOPE, COUNT_STAGE))
BASIS = "all" if VOTE_SCOPE == "all" else "ordinary"   # used in chart labels
rel = pd.read_csv(ROOT / "data/reference/electorate_relationships_2026.csv")
# Electorates are compared by name, using the Commission's list of 2025 name changes for the 10 renamed seats
# (`electorates.PREDECESSOR`, e.g. East Coast -> East Cape). The boundary review moved some of them, so `overlap`
# (share of the 2026 seat's votes from the 2023 seat it is compared with, notebook 02) is shown as a caveat.
import electorates
pv = electorates.align_2023(pv)
names = pv.groupby("election_year").electorate.agg(set)
comparable = names[2023] & names[2026]
roll = rel.set_index("electorate_2026").roll
overlap = electorates.seat_overlap()

## 1. Party share by electorate and year

In [ ]:
t = pv.groupby(["election_year", "electorate", "party"], as_index=False)["votes"].sum()
t["valid"] = t.groupby(["election_year", "electorate"]).votes.transform("sum")
t["share"] = t.votes / t.valid * 100
share = t.pivot_table(index=["electorate", "party"], columns="election_year", values="share")
share["swing_pts"] = share[2026] - share[2023]
share.loc[~share.index.get_level_values("electorate").isin(comparable), "swing_pts"] = np.nan
print(f"{len(comparable)} of {len(rel)} electorates compared across years (by name, renames via the Commission list)")

## 2. Where is `FOCUS` strongest in 2026?

In [ ]:
f = share.xs(FOCUS, level="party").reset_index()
f["roll"] = f.electorate.map(roll)
f["comparable"] = f.electorate.isin(comparable)
f["overlap"] = f.electorate.map(overlap)
top = f.sort_values(2026, ascending=False).head(TOP_N)
display(top.round(1))
ax = top.set_index("electorate")[2026].iloc[::-1].plot.barh(figsize=(8, 6), color="#2a9d8f")
ax.set_xlabel(f"{FOCUS} share of {BASIS} party votes, 2026 (%)")
ax.set_title(f"{FOCUS}: strongest {TOP_N} electorates, 2026")
plt.tight_layout(); plt.show()

## 3. Change since 2023 (comparable electorates)

In [ ]:
sw = f[f.comparable].dropna(subset=["swing_pts"]).sort_values("swing_pts")
ends = pd.concat([sw.head(TOP_N // 2), sw.tail(TOP_N // 2)])
ax = ends.set_index("electorate")["swing_pts"].plot.barh(figsize=(8, 6), color=np.where(ends.swing_pts > 0, "#2a9d8f", "#c1666b"))
ax.set_xlabel("Change in share, 2023 → 2026 (percentage points)")
ax.set_title(f"{FOCUS}: biggest falls and rises by electorate")
plt.tight_layout(); plt.show()
display(sw[["electorate", "roll", 2023, 2026, "swing_pts"]].round(1).tail(5))

## 4. Leading party in each electorate (party vote)
`flipped` marks comparable electorates where a different party led in 2023.

In [ ]:
lead = t.sort_values("votes", ascending=False).drop_duplicates(["election_year", "electorate"])
lead = lead.pivot(index="electorate", columns="election_year", values="party").rename(columns={2023: "lead_2023", 2026: "lead_2026"})
margin = (t.sort_values("share", ascending=False).groupby(["election_year", "electorate"]).share
          .apply(lambda s: s.iloc[0] - s.iloc[1]).unstack(level=0).rename(columns={2023: "margin_2023", 2026: "margin_2026"}))
lead = lead.join(margin)
lead.loc[~lead.index.isin(comparable), ["lead_2023", "margin_2023"]] = np.nan
lead["flipped"] = lead.lead_2023.notna() & (lead.lead_2023 != lead.lead_2026)
print("Electorates led by each party in 2026:")
display(lead.lead_2026.value_counts().to_frame("electorates"))
display(lead[lead.flipped].round(1))
display(lead.sort_values("margin_2026").head(10).round(1))   # closest contests for the party vote

## 4b. Seat winners (candidate vote) vs party-vote leaders, 2026
Electorate seats go to the candidate with most **candidate** votes, which is not always the party with most
**party** votes. This reproduces the Commission's projected electorate seats (National 42, Labour 21, Green 2, ACT 1,
Te Pāti Māori 5); the same check runs in `tests/official_results/test_2026_national.py`.
The candidate → party link for winners is in `data/reference/ge2026_winner_parties.csv`.

In [ ]:
import electorates

cv = vote_scope.load("stg_candidate_votes", VOTE_SCOPE, COUNT_STAGE)
win = electorates.winners_2026(cv, kinds=None, stage=COUNT_STAGE).set_index("electorate")
win["party_vote_leader"] = lead.lead_2026
win["differs"] = win.party != win.party_vote_leader
display(win.party.value_counts().to_frame("seats won"))
print(f"{win.differs.sum()} of {len(win)} seats were won by a different party than led the party vote:")
display(win[win.differs][["candidate", "party", "party_vote_leader", "margin"]])
print(f"Narrowest seat margins ({BASIS}" + ("; special votes can still change these)" if VOTE_SCOPE == "ordinary" else ")"))
display(win.sort_values("margin").head(8)[["candidate", "party", "margin"]])

## 5. The shape of each electorate: top parties, 2026

In [ ]:
big = t[t.election_year == 2026].groupby("party").votes.sum().nlargest(HEATMAP_PARTIES).index
h = t[(t.election_year == 2026) & t.party.isin(big)].pivot(index="electorate", columns="party", values="share")[list(big)]
h = h.sort_values(FOCUS if FOCUS in h else big[0], ascending=False)
fig, ax = plt.subplots(figsize=(8, 18))
im = ax.imshow(h.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(h.columns)), h.columns, rotation=45, ha="right")
ax.set_yticks(range(len(h)), h.index, fontsize=7)
fig.colorbar(im, label=f"share of {BASIS} party votes (%)", shrink=0.5)
ax.set_title("Party share by electorate, 2026")
plt.tight_layout(); plt.show()

## 6. How many votes (comparable electorates)
Valid votes on the chosen basis (all votes in both years when `VOTE_SCOPE="all"`, ordinary votes only otherwise, which
understates 2026 until special votes are counted). A rough guide to turnout, not turnout itself: enrolment is not in the data.

In [ ]:
v = t.groupby(["election_year", "electorate"]).valid.first().unstack(level=0)
v = v[v.index.isin(comparable)].dropna()
v["change_pct"] = (v[2026] / v[2023] - 1) * 100
display(v.sort_values("change_pct").round(1).head(5)); display(v.sort_values("change_pct").round(1).tail(5))

## 7. Save the full table
One row per electorate × party, ready to open in a spreadsheet.

In [ ]:
out = share.reset_index().rename(columns={2023: "share_2023", 2026: "share_2026"})
out["roll"] = out.electorate.map(roll)
out["comparable"] = out.electorate.isin(comparable)
out.round(3).to_csv(ROOT / "data/staging/electorate_party_share.csv", index=False)
print(f"{len(out)} rows -> data/staging/electorate_party_share.csv")